# Tutorial: Synthetic Contact Data

In the process of developing statistical models, it is important to test them against synthetic datasets to which we know the ground truth. This allows us to check if our models are working in the way we intend them to, and to make fair comparisons between different modeling approaches. To create a platform for developing and testing social contact models, `cntmosaic` provides tools that allow users to generate:
1. Synthetic populations
2. Synthetic participant data
3. Contact intensity matrices
4. Synthetic contact data.
This toolkit and pipeline can be used to create a diverse array of synthetic datasets for testing, validating, and comparing models.

The aim of this tutorial is to provide detailed guidance on how to use the `sim` module within the `cntmosaic` package to generate the aforementioned synthetic datasets. It also serves as a high-level documentation for the API and how the different classes interact with each other.

## Setup

The code chunk below imports the necessary libraries and classes required for this tutorial.

In [ ]:
# Import the cntmosaic package
from cntmosaic.datasets import load_age_distribution, load_template_patterns
from cntmosaic.sim import (
    Stratification,
    PopulationConstructor,
    ParticipantGenerator,
    MatrixGenerator,
    ContactGenerator,
)

# Visualization and utility functions
from cntmosaic.vis import plot_mosaic
from cntmosaic.utils import save_tutorial_figure
import altair as alt

## Generating a Synthetic Population

To generate a realistic (but fake) population, it is easiest to start with a real population. The `load_age_distribution` function can be used to load the (historical) age distribution of a given country from the built-in `cntmosaic` datasets. In this example, we will load the age distribution for the United States. 

In [ ]:
df_ref_pop = load_age_distribution("United_States", max_age=80)

`load_age_distribution` will always return a pandas DataFrame with column `age` and `P` (population size). We can quickly visualize this distribution by running the chunk below.

In [ ]:
chart_ref_pop = (
    alt.Chart(df_ref_pop)
    .mark_bar()
    .encode(x=alt.X("age:O", title="Age"), y=alt.Y("P:Q", title="Population Size"))
)
chart_ref_pop.properties(
    title="Reference Population Age Distribution (United States)", width=800
)

### Dividing the Population

In reality, populations can be divided into various strata based on different characteristics. Each stratum may have its own unique age distribution. For example, the age distribution of low income, middle income, and high income groups may differ significantly. Before we can start to split the population into strata, we first need to define the strata. This is what the `Stratification` class is for.

In [ ]:
strata = Stratification(
    name="income",
    n_strata=3,
    ref_age_dist=df_ref_pop["P"].values,
    labels=["low", "middle", "high"],  # Optional but recommended
    seed=0,
)

To define a stratification, we need 3 things:
1. A name for the stratification (e.g., "income")
2. The number of strata (e.g., 3 for low, middle, high)
3. A reference age distribution (e.g., the age distribution of the entire population loaded earlier)
Optionally, we can also provide labels for each stratum to make it easier to interpret later on (recommended).

In [ ]:
pop_const = PopulationConstructor(strata)
df_pop = pop_const.df_P

In [ ]:
chart_pop = (
    alt.Chart(df_pop)
    .mark_bar()
    .encode(
        x=alt.X("age:O", title="Age"),
        y=alt.Y("P:Q", title="Population Size"),
        color=alt.Color("income:N", title="Income Stratum"),
    )
).properties(width=800, title="Stratified Population Age Distribution")

chart_pop

## Generating Participant Data

The next step is to generate synthetic participant data based on the stratified population. For simplicity, we will assume that the survey is a representative random sample of the population. This means that the age and stratum distribution of the participants should match that of the overall population. To generate participant data, we use the `ParticipantGenerator` class.

In [ ]:
part_gen = ParticipantGenerator(popcon=pop_const, n_part=1500)
df_part = part_gen.generate(seed=0)
df_part.head(5)

## Generating Contact Matrices

In [ ]:
templates = load_template_patterns("United_States", smooth=True, max_age=80)

matrix_gen = MatrixGenerator(templates)
cint_matrices = matrix_gen.generate_partial(popcon=pop_const, seed=2)

for key, matrix in cint_matrices.items():
    print(f"[{key}] Average contact intensity: {matrix.mean():.4f}")

In [ ]:
chart_list = [
    plot_mosaic(matrix, title=label, zlabel="Intensity")
    for label, matrix in cint_matrices.items()
]
alt.hconcat(*chart_list).properties(title="Contact Matrices by Income")

## Generate Contact Data

In [ ]:
cnt_gen = ContactGenerator(df_part, cint_matrices, random_effects=True)
df_cnt = cnt_gen.generate(seed=0)
df_cnt.head(5)

## Generating Full Scenarios

In [ ]:
cint_matrices_full = matrix_gen.generate_full(pop_const, 1, seed=2)
for key, matrix in cint_matrices_full.items():
    print(f"[{key}] Average contact intenstity: {matrix.mean():.4f}")

In [ ]:
chart_list = []
for s in ["low->low", "low->middle", "low->high"]:
    chart = plot_mosaic(cint_matrices_full[s], title=s, zlabel="Intensity")
    chart_list.append(chart)

alt.hconcat(*chart_list).properties(
    title="Full Contact Matrices for Low Income Stratum"
)

In [ ]:
chart_list = []
for s in ["middle->low", "middle->middle", "middle->high"]:
    chart = plot_mosaic(cint_matrices_full[s], title=s, zlabel="Intensity")
    chart_list.append(chart)

alt.hconcat(*chart_list).properties(
    title="Full Contact Matrices for Middle Income Stratum"
)

In [ ]:
chart_list = []
for s in ["high->low", "high->middle", "high->high"]:
    chart = plot_mosaic(cint_matrices_full[s], title=s, zlabel="Intensity")
    chart_list.append(chart)

alt.hconcat(*chart_list).properties(
    title="Full Contact Matrices for High Income Stratum"
)

## Understanding the Basics

### What is a Contact Intensity Matrix?
A contact intensity matrix $\mathbf{M}$ is a $A \times A$ matrix where:
- $A$ is the number of age groups.
- Each entry $m_{ij}$ represents the expected contact intensity between individuals in age group $i$ and age group $j$.
- Higher values indicate more frequent contacts between the respective age groups.

### Marginal contact intensity
The marginal contact intensity for an age group $i$ is defined as the sum of contact intensities across all age groups:
$$
m_i = \sum_{j=1}^{A} m_{i,j}.
$$

### What is the Subgroup Class?

The `Subgroup` dataclass packages together all the information needed to define a population subgroup:
```python
@dataclass
class Subgroup:
  n: int                      # Number of participants in this subgroup
  age_dist: NDArray           # Age distribution for this subgroup
  mean_cint_margin: NDArray   # Average marginal contact intensity for this subgroup
  label: str = 'subgroup'     # Label for this subgroup
```

---

## Part 1: Simple Single Population Example
We start with the simplest case: a single homogenous population.

### Step 1: Load Contact Pattern Templates
First, load-precomputed contact pattern templates for a country:

In [ ]:
templates = load_template_patterns(country="United_States")
print(templates.keys())

Templates contain patterns for 4 contexts:
- household: family contacts at home
- school: contacts at educational institutions
- work: contacts at workplaces
- community: all other contacts (community, leisure, transport, etc.)

We visualise the template for each setting in the following figure:

In [ ]:
charts = []
for i, (key, pattern) in enumerate(templates.items()):
    if i != 0:
        chart = plot_mosaic(pattern, title=key.capitalize(), ylabel=None)
    else:
        chart = plot_mosaic(pattern, title=key.capitalize())
    charts.append(chart)

chart = alt.hconcat(*charts).resolve_scale(color="independent")
save_tutorial_figure(
    chart, "tutorial_gen_cnt_data_template_patterns", format="svg"
)  # Save figure for documentation
chart  # display

For convenience, we will also load the population age distribution for the United States. The population distribution can be set freely as long as the dimensions match that of the contact pattern templates.

In [ ]:
df_age_dist = load_age_distribution("United_States")
age_dist = df_age_dist.P.values  # Extract as an numpy array

### Step 2: Define Your Population
Create a subgroup with the age distribution:

In [ ]:
my_population = Subgroup(
    n=1000,  # Generate 1000 participants
    age_dist=age_dist,  # Use this age distribution
    mean_cint_margin=15.0,  # Average 15 contacts per person
    label="general",
)

### Step 3. Generate Contact Matrix

In [ ]:
# Initialize the matrix generator
matrix_gen = MatrixGenerator(templates)

# Generate a contact intensity matrix
contact_matrix = matrix_gen.generate_single(
    subgroup=my_population, seed=42  # For reproducibility
)

print("Matrix shape:", contact_matrix.shape)
print("Average contacts per person:", contact_matrix.sum(axis=0).mean())

In [ ]:
chart = plot_mosaic(
    contact_matrix, title="Generated Contact Matrix", zlabel="Intensity"
)
save_tutorial_figure(
    chart, "tutorial_gen_cnt_data_generated_contact_matrix", format="svg"
)  # Save figure for documentation
chart  # display

### Step 4: Generate Participants

In [ ]:
# Initialize the participant generator
part_gen = ParticipantGenerator(my_population)

# Generate participants
df_part = part_gen.generate(seed=42)

print(df_part.head())

Each participant has:
- `id`: Unique identifier
- `age_group`: Age group index (0-81) in this example

### Step 5: Generate Contacts

In [ ]:
cnt_gen = ContactGenerator(
    df_part=df_part, cint_matrices=contact_matrix, model="poisson"
)

df_cnt = cnt_gen.generate(seed=42)

print(f"Generated {len(df_cnt)} contacts")
print(df_cnt.head())

In [ ]:
emp_cint_matrix = cnt_gen.get_contact_matrix_empirical(df_cnt, normalize=True)
chart = plot_mosaic(
    emp_cint_matrix, title="Empirical Contact Matrix", zlabel="Intensity"
)
save_tutorial_figure(
    chart, "tutorial_gen_cnt_data_empirical_contact_matrix", format="svg"
)  # Save figure for documentation
chart  # display

---
## Part 2: Multiple Subgroups (Partial Case)

Now let's model a population with distinct subgroups, such as urban vs. rural populations.

### Step 1: Define Multiple Subgroups

In [ ]:
urban_age_dist = age_dist * 0.7  # Urban population
rural_age_dist = age_dist * 0.3  # Rural population

# Create subgroup configurations
subgroups = [
    Subgroup(
        n=1000,
        age_dist=urban_age_dist,
        mean_cint_margin=10.0,  # Urban: more contacts
        label="urban",
    ),
    Subgroup(
        n=500,
        age_dist=rural_age_dist,
        mean_cint_margin=5.0,  # Rural: fewer contacts
        label="rural",
    ),
]

### Step 2: Generate Contact Matrices (Partial Case)

In the **partial case**, each subgroup has its own contact pattern with the general population:

In [ ]:
matrix_gen = MatrixGenerator(templates)
contact_matrices = matrix_gen.generate_partial(subgroups, seed=42)

# Access individual matrices
urban_contacts = contact_matrices["urban"]  # Urban subgroup matrix
rural_contacts = contact_matrices["rural"]  # Rural subgroup matrix

print(f"Urban matrix shape: {urban_contacts.shape}")
print(f"Rural matrix shape: {rural_contacts.shape}")
print(f"Urban avg intensity: {urban_contacts.sum(axis=0).mean():.2f}")
print(f"Rural avg intensity: {rural_contacts.sum(axis=0).mean():.2f}")

In [ ]:
plt_urban = plot_mosaic(
    urban_contacts, title="Urban Contact Matrix", zlabel="Intensity"
)
plt_rural = plot_mosaic(
    rural_contacts, title="Rural Contact Matrix", zlabel="Intensity"
)
chart = alt.hconcat(plt_urban, plt_rural).resolve_scale(color="independent")
save_tutorial_figure(
    chart, "tutorial_gen_cnt_data_urban_rural_matrices", format="svg"
)  # Save figure for documentation
chart  # display

### Step 3: Generate Participants for Multiple Subgroups

In [ ]:
# Generate participants
participant_gen = ParticipantGenerator(subgroups)
participants = participant_gen.generate(seed=42)

print(participants.head(10))
print(f"\nTotal participants: {len(participants)}")
print(participants["subgroup"].value_counts())

### Step 4: Generate Contacts for Multiple Subgroups

In [ ]:
cnt_gen = ContactGenerator(participants, contact_matrices, "poisson")
df_cnt = cnt_gen.generate(seed=42)

print(f"Generated {len(df_cnt)} contacts")
print(df_cnt.head())

In [ ]:
emp_cint_matrix = cnt_gen.get_contact_matrix_empirical(df_cnt, normalize=True)

plt_urban_emp = plot_mosaic(
    emp_cint_matrix["urban"],
    title="Empirical Contact Matrix - Urban",
    zlabel="Intensity",
)

plt_rural_emp = plot_mosaic(
    emp_cint_matrix["rural"],
    title="Empirical Contact Matrix - Rural",
    zlabel="Intensity",
)

chart = alt.hconcat(plt_urban_emp, plt_rural_emp).resolve_scale(color="independent")
save_tutorial_figure(
    chart, "tutorial_gen_cnt_data_urban_rural_empirical_matrices", format="svg"
)  # Save figure for documentation
chart  # display

---

## Part 3: Full Subgroup Interactions (Full Case)
The **full case** generates matrices for all possible subgroup pairs, modeling how different subgroups interact with each other.

### Generate Full Contact Matrices

In [ ]:
# Using the same subgroups as before
matrix_gen = MatrixGenerator(templates)
full_matrices = matrix_gen.generate_full(subgroups, seed=42)

# The full case returns a dictionary with tuple keys: (source, target)
print("Available matrices:")
for key in full_matrices.keys():
    print(f"  {key}: contacts from {key[0]} to {key[1]}")

In [ ]:
# Within-group contacts
urban_to_urban = full_matrices[("urban", "urban")]
rural_to_rural = full_matrices[("rural", "rural")]

# Between-group contacts
urban_to_rural = full_matrices[("urban", "rural")]
rural_to_urban = full_matrices[("rural", "urban")]

print(f"Urban→Urban avg: {urban_to_urban.sum(axis=0).mean():.2f}")
print(f"Rural→Rural avg: {rural_to_rural.sum(axis=0).mean():.2f}")
print(f"Urban→Rural avg: {urban_to_rural.sum(axis=0).mean():.2f}")
print(f"Rural→Urban avg: {rural_to_urban.sum(axis=0).mean():.2f}")

In [ ]:
matrices_to_plot = [
    (("urban", "urban"), "Urban → Urban"),
    (("urban", "rural"), "Urban → Rural"),
    (("rural", "urban"), "Rural → Urban"),
    (("rural", "rural"), "Rural → Rural"),
]

chart = alt.hconcat(
    *[
        plot_mosaic(full_matrices[(k, l)], title=title, zlabel="Intensity")
        for (k, l), title in matrices_to_plot
    ]
).resolve_scale(color="independent")
save_tutorial_figure(
    chart, "tutorial_gen_cnt_data_full_matrices", format="svg"
)  # Save figure for documentation
chart  # display

### Generate Contacts for Full Subgroup Interactions

In [ ]:
cnt_gen = ContactGenerator(participants, full_matrices, "poisson")
df_cnt = cnt_gen.generate(seed=42)

print(f"Generated {len(df_cnt)} contacts")
print(df_cnt.head())

In [ ]:
emp_cint_matrices = cnt_gen.get_contact_matrix_empirical(df_cnt, normalize=True)

matrices_to_plot = [
    (("urban", "urban"), "Urban → Urban"),
    (("urban", "rural"), "Urban → Rural"),
    (("rural", "urban"), "Rural → Urban"),
    (("rural", "rural"), "Rural → Rural"),
]

chart = alt.hconcat(
    *[
        plot_mosaic(emp_cint_matrices[(k, l)], title=title, zlabel="Intensity")
        for (k, l), title in matrices_to_plot
    ]
).resolve_scale(color="independent")
save_tutorial_figure(
    chart, "tutorial_gen_cnt_data_full_empirical_matrices", format="svg"
)  # Save figure for documentation
chart  # display